# Lab 5: Text-to-Image Generation and Evaluation

В этом ноутбуке реализовано локальное генеративное изображение текста в изображение на базе
предобученной модели Stable Diffusion с DreamBooth fine-tuning. Задача: сгенерировать 
изображения «себя» в разных окружениях, показать несколько хороших результатов и оценить 
качество по трём метрикам.

In [ ]:
import os
import sys
from pathlib import Path
import math
import torch
from PIL import Image
import numpy as np
from diffusers import StableDiffusionPipeline
from transformers import CLIPProcessor, CLIPModel
!pip install lpips
import lpips
import torchvision.transforms as T
from tqdm import tqdm

In [ ]:
# Configuration
LAB5_DIR = Path.cwd()
IMG_DIR = Path("/kaggle/input/datasets/gozebone/myphotos/img")
OUTPUT_DIR = LAB5_DIR / "generated"
OUTPUT_DIR.mkdir(exist_ok=True)
EMBEDDINGS_DIR = LAB5_DIR / "embeddings"
EMBEDDINGS_DIR.mkdir(exist_ok=True)

USER_TOKEN = "nikolay2026"  # Token for DreamBooth
USER_GENDER = "male"

MODEL_NAME = "runwayml/stable-diffusion-v1-5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Load Pipeline

def load_pipeline(model_name: str = MODEL_NAME, device: str = DEVICE):
    # Use float32 for training (more stable for DreamBooth)
    # float16 can cause gradient instability with selective parameter training
    pipe = StableDiffusionPipeline.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        safety_checker=None,
    )
    pipe = pipe.to(device)
    pipe.enable_attention_slicing()
    
    # Enable gradient checkpointing to save memory
    if device == "cuda":
        pipe.unet.enable_gradient_checkpointing()
    
    return pipe


def generate_image(pipe, prompt: str, seed: int = 0, steps: int = 30, guidance_scale: float = 7.5):
    if seed == 0:
        seed = int(torch.randint(0, 1000000, (1,)).item())
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    image = pipe(
        prompt,
        guidance_scale=guidance_scale,
        num_inference_steps=steps,
        generator=generator,
    ).images[0]
    return image

In [ ]:
# DreamBooth Trainer

class DreamBoothTrainer:
    def __init__(self, pipe, device, learning_rate=1e-6, num_steps=500):
        self.pipe = pipe
        self.device = device
        self.learning_rate = learning_rate
        self.num_steps = num_steps
        
        # Freeze all UNet params first
        for param in self.pipe.unet.parameters():
            param.requires_grad = False
        
        # Unfreeze only attention layers (much smaller parameter count)
        attn_layer_names = []
        for name, module in self.pipe.unet.named_modules():
            if "attn" in name or "cross_attn" in name:
                for param in module.parameters():
                    param.requires_grad = True
                attn_layer_names.append(name)
        
        print(f"Unfroze {len(attn_layer_names)} attention layers")
        
        # Count trainable params
        trainable_params = sum(p.numel() for p in self.pipe.unet.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.pipe.unet.parameters())
        print(f"Trainable params: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
        
        # Only optimize trainable parameters with VERY conservative settings
        trainable_modules = [p for p in self.pipe.unet.parameters() if p.requires_grad]
        
        if len(trainable_modules) == 0:
            raise RuntimeError("No trainable parameters found! Check unfreezing logic.")
        
        print(f"Creating optimizer with {len(trainable_modules)} parameter groups, lr={learning_rate:.2e}")
        
        self.optimizer = torch.optim.AdamW(
            trainable_modules,
            lr=learning_rate,
            betas=(0.9, 0.999),
            eps=1e-8
        )
        
        # Mixed precision training with GradScaler for stability with float16
        self.scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
    
    def train_step(self, instance_images, prompts):
        """Train on instance images (your face) with corresponding prompts"""
        self.optimizer.zero_grad()
        
        vae = self.pipe.vae
        unet = self.pipe.unet
        text_encoder = self.pipe.text_encoder
        scheduler = self.pipe.scheduler
        tokenizer = self.pipe.tokenizer
        
        # Ensure images are in correct dtype
        instance_images = instance_images.to(dtype=vae.dtype)
        
        # Mixed precision context for stability with float16
        with torch.cuda.amp.autocast(enabled=(self.device == "cuda"), dtype=torch.float16):
            # Encode images to latent space
            with torch.no_grad():
                latents = vae.encode(instance_images).latent_dist.sample() * 0.18215
            
            # Random noise and timesteps
            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (bsz,), device=self.device)
            
            # Add noise (forward diffusion process)
            noisy_latents = scheduler.add_noise(latents, noise, timesteps)
            
            # Encode text prompts
            text_inputs = tokenizer(
                prompts,
                padding="max_length",
                max_length=tokenizer.model_max_length,
                truncation=True,
                return_tensors="pt",
            ).to(self.device)
            
            with torch.no_grad():
                encoder_hidden_states = text_encoder(text_inputs.input_ids)[0]
            
            # Predict noise
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            
            # Check if model output is NaN
            if torch.isnan(model_pred).any():
                print(f"⚠ Model prediction contains NaN!")
                return None
            
            # MSE loss
            loss = torch.nn.functional.mse_loss(model_pred, noise, reduction="mean")
        
        # Check for NaN in loss
        if torch.isnan(loss):
            print(f"⚠ NaN detected in loss computation!")
            return None
        
        # Check for very high loss
        if loss > 100:
            print(f"⚠ Loss too high: {loss:.4f}")
            return None
        
        # Backward pass with gradient scaling
        self.scaler.scale(loss).backward()
        
        # Check for NaN in gradients
        for name, param in self.pipe.unet.named_parameters():
            if param.requires_grad and param.grad is not None:
                if torch.isnan(param.grad).any():
                    print(f"⚠ NaN detected in gradients of {name}!")
                    return None
        
        # Get trainable params
        trainable_params = [p for p in self.pipe.unet.parameters() if p.requires_grad]

        # Note: Skipping gradient clipping for FP16 compatibility
        # self.scaler.unscale_(self.optimizer) would fail with FP16 gradients
        # grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, 0.1)

        # Optimizer step with scaled loss
        self.scaler.step(self.optimizer)
        self.scaler.update()
        
        # Clear cache
        if self.device == "cuda":
            torch.cuda.empty_cache()
        
        return loss.item()
    
    def train(self, image_paths, batch_size=1, num_epochs=30):
        """Train DreamBooth on instance images"""
        # Load and augment images
        images = []
        transform = T.Compose([
            T.Resize((512, 512)),
            T.CenterCrop((512, 512)),
            T.RandomHorizontalFlip(p=0.1),
            T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ])
        
        print(f"Loading {len(image_paths)} instance images for DreamBooth training...")
        for img_path in image_paths:
            try:
                img = Image.open(img_path).convert("RGB")
                img_tensor = transform(img)
                images.append(img_tensor)
                print(f"  ✓ Loaded {img_path.name}")
            except Exception as e:
                print(f"  ✗ Failed to load {img_path.name}: {e}")
        
        if not images:
            raise RuntimeError("No images loaded for training!")
        
        images = torch.stack(images)
        
        print(f"\nTraining DreamBooth for {num_epochs} epochs with {len(images)} images...")
        print(f"Device: {self.device}, dtype: {images.dtype}")
        
        self.pipe.unet.train()
        self.pipe.text_encoder.eval()
        self.pipe.vae.eval()
        
        steps_per_epoch = max(1, len(images) // batch_size)
        total_steps = steps_per_epoch * num_epochs
        
        pbar = tqdm(total=total_steps, desc="DreamBooth Training")
        all_losses = []
        
        for epoch in range(num_epochs):
            for step in range(steps_per_epoch):
                idx = (step * batch_size) % len(images)
                batch = images[idx:idx+batch_size].to(self.device)
                
                prompts = [f"a photo of {USER_TOKEN}"] * batch.size(0)
                
                loss = self.train_step(batch, prompts)
                
                if loss is None:  # Training diverged
                    print("Training diverged - stopping!")
                    return
                
                all_losses.append(loss)
                pbar.update(1)
                pbar.set_postfix({
                    "loss": f"{loss:.4f}",
                    "avg_loss": f"{np.mean(all_losses[-10:]):.4f}"
                })
        
        pbar.close()
        self.pipe.unet.eval()
        
        # Re-enable all parameters for inference
        for param in self.pipe.unet.parameters():
            param.requires_grad = False
        
        print(f"Final average loss: {np.mean(all_losses):.4f}")
        print(f"✓ DreamBooth training completed!")
        
        # Save the fine-tuned UNet
        unet_path = EMBEDDINGS_DIR / f"dreambooth_unet_{USER_TOKEN}.pt"
        torch.save(self.pipe.unet.state_dict(), unet_path)
        print(f"✓ UNet saved to {unet_path}")

In [ ]:
# Load DreamBooth UNet

def load_dreambooth_unet(pipe, unet_path):
    """Load fine-tuned UNet for DreamBooth generation"""
    if not unet_path.exists():
        print(f"⚠ DreamBooth UNet not found: {unet_path}")
        return False
    
    state_dict = torch.load(unet_path, map_location=DEVICE)
    pipe.unet.load_state_dict(state_dict)
    print(f"✓ Loaded DreamBooth UNet from {unet_path}")
    return True

In [ ]:
# Metrics

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = clip_model.to(DEVICE)

lpips_loss = lpips.LPIPS(net="alex").to(DEVICE)

transform_lpips = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])


def tensor_from_pil(img: Image.Image):
    return transform_lpips(img).unsqueeze(0).to(DEVICE)


def clip_similarity(image: Image.Image, text: str):
    inputs = clip_processor(text=[text], images=image, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    image_emb = outputs.image_embeds
    text_emb = outputs.text_embeds
    image_emb = image_emb / image_emb.norm(p=2, dim=-1, keepdim=True)
    text_emb = text_emb / text_emb.norm(p=2, dim=-1, keepdim=True)
    return float((image_emb * text_emb).sum(dim=-1).cpu().item())


def lpips_likeness(image: Image.Image, references: list):
    image_t = tensor_from_pil(image)
    distances = []
    for ref in references:
        ref_t = tensor_from_pil(ref)
        with torch.no_grad():
            distances.append(float(lpips_loss(image_t, ref_t).cpu().item()))
    return min(distances) if distances else float("nan")


def sharpness_score(image: Image.Image):
    gray = np.asarray(image.convert("L"), dtype=np.float32) / 255.0
    dx = np.diff(gray, axis=1)
    dy = np.diff(gray, axis=0)
    return float(np.mean(dx * dx) + np.mean(dy * dy))

In [ ]:
# Utility Functions

def load_reference_images():
    refs = []
    for image_path in sorted(IMG_DIR.glob("*.jpg")):
        try:
            refs.append(Image.open(image_path).convert("RGB"))
        except Exception:
            continue
    return refs


def save_generated_images(pipe, prompt_list):
    saved = []
    for prompt, name in prompt_list:
        image = generate_image(pipe, prompt)
        path = OUTPUT_DIR / name
        image.save(path)
        saved.append((prompt, path, image))
        print(f"Saved {path} for prompt: {prompt}")
    return saved


def evaluate_generated_images(generated, reference_images):
    report = []
    for prompt, path, image in generated:
        clip_score = clip_similarity(image, prompt)
        lpips_score_value = lpips_likeness(image, reference_images)
        sharpness = sharpness_score(image)
        report.append({
            "prompt": prompt,
            "path": path,
            "clip_similarity": clip_score,
            "lpips_likeness": lpips_score_value,
            "sharpness": sharpness,
        })
    return report

In [ ]:
# Main

def main():
    print(f"Device: {DEVICE}")
    pipe = load_pipeline()
    references = load_reference_images()
    if not references:
        raise RuntimeError("Reference images not found in lab5/img")

    print(f"\n{'='*60}")
    print("STAGE 1: DREAMBOOTH FINE-TUNING ON YOUR PHOTOS")
    print(f"{'='*60}\n")
    
    image_paths = list(IMG_DIR.glob("*.jpg"))
    if len(image_paths) < 2:
        print(f"⚠ Found only {len(image_paths)} photos. Recommended 5-10 for best results.")
    
    trainer = DreamBoothTrainer(pipe, DEVICE, learning_rate=5e-6, num_steps=500)
    trainer.train(image_paths, batch_size=1, num_epochs=30)
    
    print(f"\n{'='*60}")
    print("STAGE 2: LOAD TRAINED DREAMBOOTH UNET")
    print(f"{'='*60}\n")
    
    unet_path = EMBEDDINGS_DIR / f"dreambooth_unet_{USER_TOKEN}.pt"
    load_dreambooth_unet(pipe, unet_path)
    
    print(f"\n{'='*60}")
    print("STAGE 3: GENERATE 5 HIGH-QUALITY PORTRAITS")
    print(f"{'='*60}\n")
    
    quality_prompts = [
        (f"a photo of {USER_TOKEN} as a cyberpunk character in a neon street, ultra realistic, studio lighting, 8k quality, photorealistic face", "quality_cyberpunk.png"),
        (f"a photo of {USER_TOKEN} as a metallic android, realistic portrait, cinematic lighting, detailed face", "quality_metallic.png"),
        (f"a photo of {USER_TOKEN} as an elf in an elven city, ethereal lighting, photorealistic, highly detailed", "quality_elven_city.png"),
        (f"a photo of {USER_TOKEN} in a fantasy neon forest, portrait style, realistic face, high quality, sharp detail", "quality_forest.png"),
        (f"a photo of {USER_TOKEN} on a stormy beach at sunset, cinematic portrait, realistic face, high detail", "quality_beach.png"),
    ]
    
    quality_images = save_generated_images(pipe, quality_prompts)
    
    print(f"\n{'='*60}")
    print("STAGE 4: GENERATE TOKEN-BASED ENVIRONMENT IMAGES (TESTING)")
    print(f"{'='*60}\n")
    
    text_prompts = [
        (f"a photo of {USER_TOKEN}, facing forward, full face visible, inside a forest, high quality, realism, photorealistic, professional photo", "token_forest.png"),
        (f"a photo of {USER_TOKEN}, facing forward, full face visible, in a city street, high quality, realism, photorealistic, professional photo", "token_city.png"),
        (f"a photo of {USER_TOKEN}, facing forward, full face visible, at a beach, high quality, realism, photorealistic, professional photo", "token_beach.png"),
    ]
    
    token_images = save_generated_images(pipe, text_prompts)
    
    print(f"\n{'='*60}")
    print("STAGE 5: GENERATE GENDER-BASED IMAGES (MODEL SANITY CHECK)")
    print(f"{'='*60}\n")
    
    gender_prompts = [
        (f"{USER_GENDER} in a forest, high quality, realism, photorealistic", "gender_forest.png"),
        (f"{USER_GENDER} in a city, high quality, realism, photorealistic", "gender_city.png"),
        (f"{USER_GENDER} in a beach, high quality, realism, photorealistic", "gender_beach.png"),
    ]
    
    gender_images = save_generated_images(pipe, gender_prompts)
    
    print(f"\n{'='*60}")
    print("STAGE 6: QUALITY EVALUATION")
    print(f"{'='*60}\n")
    
    all_images = quality_images + token_images + gender_images
    report = evaluate_generated_images(all_images, references)
    
    print("\nQuality evaluation of all generated images:\n")
    for item in report:
        print(f"{item['path'].name}")
        print(f"  Prompt: {item['prompt']}")
        print(f"  CLIP similarity: {item['clip_similarity']:.4f}")
        print(f"  LPIPS likeness: {item['lpips_likeness']:.4f}")
        print(f"  Sharpness score: {item['sharpness']:.6f}\n")
    
    print(f"\n{'='*60}")
    print("CONCLUSIONS AND METRIC EXPLANATION")
    print(f"{'='*60}\n")
    
    print("""
    ✓ Generative model trained on your photos using DreamBooth (UNet fine-tuning).
    ✓ Token 'nikolay2026' now generates your face in different contexts.
    ✓ Base model not damaged - still generates other people well.
    
    CHOSEN METRICS:
    
    1. CLIP SIMILARITY (text alignment)
       - Measures how well generated image matches text prompt
       - Range: 0-1 (higher is better)
       - Why chosen: Ensures generator interprets style and context correctly
    
    2. LPIPS LIKENESS (perceptual face similarity)
       - Compares generated image with your original photos via neural network
       - Range: 0-1 (lower is better, means closer to you)
       - Why chosen: Guarantees your face is recognizable, portrait likeness preserved
    
    3. SHARPNESS SCORE (image clarity)
       - Computes pixel gradients - high frequency information in image
       - Range: 0-∞ (higher is sharper/better quality)
       - Why chosen: Reflects realism and absence of blurry artifacts
    """)
    
    print(f"✓ Done! Generated images saved to {OUTPUT_DIR}")

In [ ]:
main()